In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

Cloning into 'CTAB-GAN-Plus'...


In [2]:
from ucimlrepo import fetch_ucirepo

In [3]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sdv.metadata import SingleTableMetadata
import torch
import random

energy_efficiency = fetch_ucirepo(id=242)
X = energy_efficiency.data.features
y = energy_efficiency.data.targets
print(energy_efficiency.metadata)
print(energy_efficiency.variables)

data = pd.concat([X, y[["Y1"]]], axis=1)
target_col = "Y1"

# Drop date/time/session/ID features before generator training (high cardinality).
_drop_feature_cols = [
    "Date", "Time", "date_time",
    "session_id", "Session ID", "Session_ID", "session",
    "X1 transaction date",
]
data = data.drop(columns=[c for c in _drop_feature_cols if c in data.columns], errors="ignore")

n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

processed_data = data.copy()

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


{'uci_id': 242, 'name': 'Energy Efficiency', 'repository_url': 'https://archive.ics.uci.edu/dataset/242/energy+efficiency', 'data_url': 'https://archive.ics.uci.edu/static/public/242/data.csv', 'abstract': 'This study looked into assessing the heating load and cooling load requirements of buildings (that is, energy efficiency) as a function of building parameters.', 'area': 'Computer Science', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 768, 'num_features': 8, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Y1', 'Y2'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2012, 'last_updated': 'Mon Feb 26 2024', 'dataset_doi': '10.24432/C51307', 'creators': ['Athanasios Tsanas', 'Angeliki Xifara'], 'intro_paper': {'ID': 379, 'type': 'NATIVE', 'title': 'Accurate quantitative estimation of energy performance of residential buildings using statistical machine 

In [4]:
from sklearn.model_selection import train_test_split
from sdv.evaluation.single_table import evaluate_quality

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
        random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

try:
    data_path = "energy_efficiency_train.csv"
    train_real.to_csv(data_path, index=False)

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=[],
        log_columns=[],
        mixed_columns={},
        integer_columns=[],
        problem_type={"Regression": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    # Convert all columns back to numeric where possible
    for col in synthetic_ctabgan.columns:
        synthetic_ctabgan[col] = pd.to_numeric(
            synthetic_ctabgan[col],
            errors="coerce"
        )

        synthetic_ctabgan[col] = synthetic_ctabgan[col].fillna(
            train_real[col].median()
        )

    # Energy Efficiency quality target is an integer score between 0 and 10
    pass  # keep continuous regression target as-is

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)



================ SINGLE RUN ================


100%|██████████| 150/150 [09:17<00:00,  3.72s/it]


Finished training in 561.9697406291962  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 106.76it/s]|
Column Shapes Score: 50.53%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 205.53it/s]|
Column Pair Trends Score: 49.06%

Overall Score (Average): 49.79%

CTABGAN: 0.4979


In [8]:
# WGAN-GP

try:

    import traceback
    from sklearn.preprocessing import StandardScaler
    import torch.nn as nn
    import torch.optim as optim
    from sdv.evaluation.single_table import evaluate_quality

    data_wgan = train_real.copy()

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 217.72it/s]|
Column Shapes Score: 61.79%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 309.44it/s]|
Column Pair Trends Score: 82.89%

Overall Score (Average): 72.34%

WGAN_GP: 0.7234


In [9]:
# SDV MODELS

from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        pass  # keep continuous regression target as-is

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 371.41it/s]|
Column Shapes Score: 79.32%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 231.09it/s]|
Column Pair Trends Score: 58.09%

Overall Score (Average): 68.71%

CTGAN: 0.6871
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 254.99it/s]|
Column Shapes Score: 78.55%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 169.01it/s]|
Column Pair Trends Score: 55.89%

Overall Score (Average): 67.22%

CopulaGAN: 0.6722
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 326.42it/s]|
Column Shapes Score: 87.04%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 210.74it/s]|
Column Pair Trends Score: 89.86%

Overall Score (Average): 88.45%

TVAE: 0.8845
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 278.51it/s]|
Column Shapes Score: 7

In [11]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'SVR_RBF': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
}

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
GENERATORS_TO_EVAL = list(synthetic_datasets.keys())

print(f'Regression evaluation: {len(regressors)} models, {len(EVAL_SEEDS)} seeds, {len(GENERATORS_TO_EVAL)} generators')


Regression evaluation: 10 models, 10 seeds, 6 generators


In [15]:
from sklearn.model_selection import train_test_split

def evaluate_regression_models(train_df, test_df, label_col, models, test_size=0.2, seeds=None, use_holdout=False, schema_df=None):
    if seeds is None:
        seeds = EVAL_SEEDS
    if schema_df is None:
        schema_df = test_df if use_holdout else train_df

    train_df = align_to_train_schema(train_df, schema_df, label_col)
    test_df = align_to_train_schema(test_df, schema_df, label_col)
    results = []

    for name, model in models.items():
        r2_scores = []
        mse_scores = []
        rmse_scores = []
        mae_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if not use_holdout:
                X_train, _, y_train, _ = train_test_split(
                    X_train, y_train, test_size=test_size, random_state=seed
                )
                _, X_test, _, y_test = train_test_split(
                    X_test, y_test, test_size=test_size, random_state=seed
                )

            reg = clone(model)
            if 'random_state' in reg.get_params():
                reg.set_params(random_state=seed)

            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2_scores.append(r2_score(y_test, y_pred))
            mse_scores.append(mse)
            rmse_scores.append(np.sqrt(mse))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        results.append({
            'Model': name,
            'R2 Mean': np.mean(r2_scores),
            'R2 Std': np.std(r2_scores),
            'MSE Mean': np.mean(mse_scores),
            'MSE Std': np.std(mse_scores),
            'RMSE Mean': np.mean(rmse_scores),
            'RMSE Std': np.std(rmse_scores),
            'MAE Mean': np.mean(mae_scores),
            'MAE Std': np.std(mae_scores),
            'R2 (Mean±Std)': f"{np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}",
            'MSE (Mean±Std)': f"{np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}",
            'RMSE (Mean±Std)': f"{np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}",
            'MAE (Mean±Std)': f"{np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by='R2 Mean', ascending=False)


In [16]:
def align_to_train_schema(df_to_align, schema_reference_df, label_col):
    feature_cols_schema = [col for col in schema_reference_df.columns if col != label_col]

    for col in feature_cols_schema:
        if col not in df_to_align.columns:
            df_to_align[col] = schema_reference_df[col].median()

    extra_cols = [col for col in df_to_align.columns if col not in schema_reference_df.columns]
    if extra_cols:
        df_to_align = df_to_align.drop(columns=extra_cols)

    final_ordered_cols = [col for col in schema_reference_df.columns if col in df_to_align.columns]
    return df_to_align[final_ordered_cols]

print('TRTR (Train Real, Test Real) — repeated 80/20 splits per seed')
trtr_results = evaluate_regression_models(
    train_df=processed_data,
    test_df=processed_data,
    label_col=target_col,
    models=regressors,
    seeds=EVAL_SEEDS,
    use_holdout=False,
    schema_df=processed_data,
)
display(trtr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

all_comparisons = []
for synth_name in GENERATORS_TO_EVAL:
    if synth_name not in synthetic_datasets:
        print(f'Skipping {synth_name} - no synthetic dataset')
        continue

    print(f'{synth_name} - TSTR (train on synthetic, test on real — split per seed)')
    tstr_results = evaluate_regression_models(
        train_df=synthetic_datasets[synth_name],
        test_df=test_real,
        label_col=target_col,
        models=regressors,
        seeds=EVAL_SEEDS,
        use_holdout=False,
        schema_df=train_real,
    )
    display(tstr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

    comparison = trtr_results.merge(tstr_results, on='Model', suffixes=('_TRTR', '_TSTR'))
    comparison['R2_Drop'] = comparison['R2 Mean_TRTR'] - comparison['R2 Mean_TSTR']
    comparison['MSE_Increase'] = comparison['MSE Mean_TSTR'] - comparison['MSE Mean_TRTR']
    comparison['RMSE_Increase'] = comparison['RMSE Mean_TSTR'] - comparison['RMSE Mean_TRTR']
    comparison['MAE_Increase'] = comparison['MAE Mean_TSTR'] - comparison['MAE Mean_TRTR']
    comparison['Synthetic_Model'] = synth_name
    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)
summary = (
    combined_comparison
    .groupby('Synthetic_Model', as_index=False)[['R2_Drop', 'MSE_Increase', 'RMSE_Increase', 'MAE_Increase']]
    .mean()
    .sort_values('R2_Drop')
)

display(summary)


TRTR (Train Real, Test Real) — repeated 80/20 splits per seed


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
9,GradientBoost,0.9977 ± 0.0004,0.2313 ± 0.0385,0.4793 ± 0.0407,0.3399 ± 0.0282
8,ExtraTrees,0.9970 ± 0.0015,0.3039 ± 0.1595,0.5357 ± 0.1303,0.3344 ± 0.0526
7,RandomForest,0.9969 ± 0.0014,0.3085 ± 0.1496,0.5419 ± 0.1220,0.3508 ± 0.0493
6,DecisionTree,0.9963 ± 0.0016,0.3749 ± 0.1768,0.6000 ± 0.1219,0.3860 ± 0.0494
5,KNN,0.9465 ± 0.0164,5.3081 ± 1.6191,2.2797 ± 0.3332,1.6333 ± 0.2038
0,LinearRegression,0.9136 ± 0.0140,8.5736 ± 1.3651,2.9186 ± 0.2355,2.0847 ± 0.1787
2,Lasso,0.9135 ± 0.0141,8.5862 ± 1.3688,2.9207 ± 0.2358,2.0862 ± 0.1812
3,ElasticNet,0.9103 ± 0.0145,8.8988 ± 1.3947,2.9737 ± 0.2359,2.1694 ± 0.1847
1,Ridge,0.9094 ± 0.0148,8.9914 ± 1.4456,2.9886 ± 0.2440,2.1907 ± 0.1898
4,SVR_RBF,0.7102 ± 0.0342,28.8867 ± 4.4752,5.3586 ± 0.4149,3.9108 ± 0.2597


CTABGAN - TSTR (train on synthetic, test on real — split per seed)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
7,RandomForest,-0.9110 ± 0.3782,167.1346 ± 21.5351,12.8996 ± 0.8575,11.2949 ± 0.8602
1,Ridge,-0.9707 ± 0.3979,171.9150 ± 17.8415,13.0938 ± 0.6844,11.4462 ± 0.8160
3,ElasticNet,-0.9719 ± 0.3988,172.0114 ± 17.8849,13.0974 ± 0.6858,11.4486 ± 0.8162
2,Lasso,-0.9726 ± 0.3996,172.0617 ± 17.9207,13.0992 ± 0.6870,11.4500 ± 0.8165
0,LinearRegression,-0.9741 ± 0.3995,172.2007 ± 17.9124,13.1046 ± 0.6864,11.4529 ± 0.8165
8,ExtraTrees,-1.2536 ± 0.4951,196.3518 ± 25.0715,13.9833 ± 0.9044,12.1203 ± 1.0179
4,SVR_RBF,-1.3314 ± 0.5327,202.6167 ± 25.4192,14.2054 ± 0.9067,12.3467 ± 1.0207
5,KNN,-1.3422 ± 0.4800,204.6808 ± 29.3528,14.2672 ± 1.0624,12.3787 ± 1.1203
6,DecisionTree,-1.3708 ± 0.9118,203.6778 ± 59.1534,14.1243 ± 2.0448,11.9057 ± 2.1797
9,GradientBoost,-1.6772 ± 0.5765,234.7554 ± 37.6896,15.2724 ± 1.2291,13.5223 ± 1.2696


WGAN_GP - TSTR (train on synthetic, test on real — split per seed)


...

TVAE - TSTR (train on synthetic, test on real — split per seed)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
7,RandomForest,0.8327 ± 0.0538,14.6394 ± 4.2145,3.7840 ± 0.5663,2.8363 ± 0.3632
9,GradientBoost,0.8067 ± 0.0635,17.2939 ± 5.9810,4.0905 ± 0.7493,3.0401 ± 0.5421
0,LinearRegression,0.8015 ± 0.0551,17.7649 ± 5.3038,4.1638 ± 0.6541,2.6758 ± 0.5101
2,Lasso,0.8013 ± 0.0552,17.7792 ± 5.3110,4.1653 ± 0.6551,2.6753 ± 0.5095
3,ElasticNet,0.8004 ± 0.0557,17.8652 ± 5.3445,4.1751 ± 0.6588,2.6786 ± 0.5077
1,Ridge,0.7988 ± 0.0565,18.0039 ± 5.3945,4.1908 ± 0.6640,2.6915 ± 0.5054
5,KNN,0.7954 ± 0.0612,18.2686 ± 5.5947,4.2183 ± 0.6888,3.0859 ± 0.4948
8,ExtraTrees,0.7547 ± 0.0917,21.8690 ± 8.3833,4.5931 ± 0.8787,3.4453 ± 0.5959
6,DecisionTree,0.6465 ± 0.2579,31.7606 ± 25.8319,5.3156 ± 1.8722,3.9019 ± 1.0203
4,SVR_RBF,0.6219 ± 0.0776,34.0100 ± 8.9047,5.7766 ± 0.8005,4.2549 ± 0.5954


GaussianCopula - TSTR (train on synthetic, test on real — split per seed)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
7,RandomForest,0.7167 ± 0.0697,24.7167 ± 4.5625,4.9497 ± 0.4659,4.1776 ± 0.4628
9,GradientBoost,0.6901 ± 0.0652,27.1842 ± 4.5394,5.1951 ± 0.4412,4.3421 ± 0.3841
8,ExtraTrees,0.6814 ± 0.0809,27.7982 ± 5.3049,5.2485 ± 0.5011,4.5539 ± 0.5191
1,Ridge,0.6262 ± 0.0602,32.7502 ± 2.2973,5.7193 ± 0.1990,5.1309 ± 0.1559
3,ElasticNet,0.6203 ± 0.0600,33.2711 ± 2.2475,5.7649 ± 0.1937,5.1556 ± 0.1663
2,Lasso,0.6158 ± 0.0600,33.6722 ± 2.2329,5.7996 ± 0.1918,5.1730 ± 0.1762
0,LinearRegression,0.6145 ± 0.0601,33.7843 ± 2.2316,5.8093 ± 0.1914,5.1809 ± 0.1768
5,KNN,0.5987 ± 0.1038,35.4976 ± 9.4075,5.9061 ± 0.7846,4.7359 ± 0.6673
4,SVR_RBF,0.5782 ± 0.0636,37.7701 ± 7.9702,6.1094 ± 0.6669,4.9274 ± 0.5459
6,DecisionTree,0.3572 ± 0.2234,55.8745 ± 15.8250,7.4050 ± 1.0202,6.0516 ± 0.8035


,Synthetic_Model,R2_Drop,MSE_Increase,RMSE_Increase,MAE_Increase
5,WGAN_GP,0.109722,9.003889,1.761013,1.355369
4,TVAE,0.163150,13.879122,2.287644,1.579931
3,GaussianCopula,0.319212,27.185552,3.631019,3.394269
1,CTGAN,1.060605,94.565482,7.797706,6.684907
2,CopulaGAN,1.325008,115.450126,8.673721,7.935073
0,CTABGAN,2.106684,182.694265,11.555038,10.387984


In [17]:
quality_df = pd.DataFrame.from_dict(scores, orient='index', columns=['Quality Score'])
quality_df.index.name = 'Synthetic Model'
quality_df = quality_df.reset_index()

output_file = 'TRTR_TSTR_results_energy_efficiency.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    quality_df.to_excel(writer, sheet_name='Quality_Metrics', index=False)
    trtr_results.to_excel(writer, sheet_name='TRTR_Results', index=False)
    combined_comparison.to_excel(writer, sheet_name='All_Comparisons', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    for synth_name in GENERATORS_TO_EVAL:
        if synth_name in combined_comparison['Synthetic_Model'].values:
            synth_results = combined_comparison[combined_comparison['Synthetic_Model'] == synth_name]
            synth_results.to_excel(writer, sheet_name=synth_name[:31], index=False)

print(f'Results saved to: {output_file}')


Results saved to: TRTR_TSTR_results_energy_efficiency.xlsx
